In [1]:
from scipy.signal import envelope, hilbert
import scipy.io as sio
import pandas as pd
import numpy as np
import sys, os
import torch
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import Extract_Features

In [2]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

In [7]:

shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "amplitude_envelope",
    frame_size = 128,
    hop_length = 32
)

print(data.get_samples().shape)
print(data.get_labels().shape)


(3309, 1122)
(3309,)


In [8]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel="rbf", C=100)
svc.fit(data.get_samples()[:3000], data.get_labels()[:3000])
print(svc.score(data.get_samples()[3000:], data.get_labels()[3000:]))

0.7637540453074434


In [12]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.2)

loops.train(model=model, model_path="envelope.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-2, optim="adam", epochs=30)

loops.test(model_path="envelope.pth", test_loader=test_loader)

[INFO] EPOCH: 1/30
Train loss: 0.595806, Train accuracy: 0.6620
[INFO] EPOCH: 2/30
Train loss: 0.545957, Train accuracy: 0.6807
[INFO] EPOCH: 3/30
Train loss: 0.523754, Train accuracy: 0.7130
[INFO] EPOCH: 4/30
Train loss: 0.515881, Train accuracy: 0.7160
[INFO] EPOCH: 5/30
Train loss: 0.512123, Train accuracy: 0.7087
[INFO] EPOCH: 6/30
Train loss: 0.502659, Train accuracy: 0.7250
[INFO] EPOCH: 7/30
Train loss: 0.494520, Train accuracy: 0.7283
[INFO] EPOCH: 8/30
Train loss: 0.494320, Train accuracy: 0.7273
[INFO] EPOCH: 9/30
Train loss: 0.481007, Train accuracy: 0.7367
[INFO] EPOCH: 10/30
Train loss: 0.479589, Train accuracy: 0.7407
[INFO] EPOCH: 11/30
Train loss: 0.473006, Train accuracy: 0.7500
[INFO] EPOCH: 12/30
Train loss: 0.469338, Train accuracy: 0.7503
[INFO] EPOCH: 13/30
Train loss: 0.461229, Train accuracy: 0.7550
[INFO] EPOCH: 14/30
Train loss: 0.456849, Train accuracy: 0.7670
[INFO] EPOCH: 15/30
Train loss: 0.444917, Train accuracy: 0.7757
[INFO] EPOCH: 16/30
Train loss: 0.

In [ ]:
## mlp with 1 layer drop or no drop = 74%
## with 2 layer some dropout = 75-76% not much difference with 559 features

## with 1122 features goes upto 77%